In [0]:
import pandas as pd

gold_df = spark.table("causal_project.gold_household").toPandas()

gold_encoded = gold_df.copy()

gold_encoded["age_group"] = gold_encoded["age_group"].map({
    "Age Group1": 1,
    "Age Group2": 2,
    "Age Group3": 3,
    "Age Group4": 4,
    "Age Group5": 5,
    "Age Group6": 6
})

gold_encoded["income_level"] = gold_encoded["income_level"].str.replace(
    "Level", "", regex=False
).astype(int)

gold_encoded["household_size"] = gold_encoded["household_size"].map({
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5+": 5
})

gold_encoded["kid_count"] = gold_encoded["kid_count"].map({
    "None/Unknown": 0,
    "1": 1,
    "2": 2,
    "3+": 3
})

gold_encoded["home_ownership"] = gold_encoded["home_ownership"].map({
    "Renter": 0,
    "Probable Renter": 1,
    "Unknown": 2,
    "Probable Owner": 3,
    "Homeowner": 4
})

gold_encoded["marital_status"] = gold_encoded["marital_status"].map({
    "X": 0,
    "Y": 1,
    "Z": 2
})

In [0]:
import numpy as np
gold_encoded['log_outcome'] = np.log1p(gold_df['avg_daily_campaign_spend'])

In [0]:
discovery_cols = [
  'treatment',
  'log_outcome',
  'pre_avg_weekly_spend',
  'pre_avg_weekly_trips', 
  'pre_coupon_usage_rate',
  'pre_loyalty_card_rate',
  'pre_department_breadth',
  'pre_active_weeks',
  'clean_window_days',
  'age_group',
  'marital_status',
  'income_level',
  'household_size',
  'home_ownership',
  'kid_count'
]

discovery_df = gold_encoded[discovery_cols]

In [0]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

data = discovery_df.to_numpy().astype(float)
demo = [  'age_group',
  'marital_status',
  'income_level',
  'household_size',
  'home_ownership',
  'kid_count'
]
pre = [
  'pre_avg_weekly_spend',
  'pre_avg_weekly_trips', 
  'pre_coupon_usage_rate',
  'pre_loyalty_card_rate',
  'pre_department_breadth',
  'pre_active_weeks']

during = ['treatment','clean_window_days']

post = ['log_outcome']

TIER_MAP = {
    **{c:0 for c in demo},
    **{c:1 for c in pre},
    **{c:2 for c in during},
    **{c:3 for c in post}
}
def build_background_knowledge(col_names):
    nodes = [GraphNode(col) for col in col_names]
    col_to_node = {col: nodes[i] for i, col in enumerate(col_names)}
    
    bk = BackgroundKnowledge()

    for col, tier in TIER_MAP.items():
        if col in col_to_node:
            bk.add_node_to_tier(col_to_node[col], tier)

    return bk
background = build_background_knowledge(discovery_df.columns.to_list())

cg = pc(data, alpha=0.05, indep_test=fisherz,node_names= discovery_df.columns.to_list(),background_knowledge=background)

print(cg.G)